In [1]:
!nvidia-smi

Sun Mar  8 12:45:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.126.09             Driver Version: 580.126.09     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        Off |   00000000:06:10.0 Off |                  N/A |
|  0%   38C    P8             15W /  350W |       1MiB /  24576MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from unsloth import FastModel
from tqdm.auto import tqdm
from unsloth.chat_templates import get_chat_template
from diffusers import StableDiffusionXLPipeline
import matplotlib.pyplot as plt
import random
from PIL import ImageFont
import torch

from MonsterNameGenerator import MarkovMonsterNameGenerator
from textGenerateUtils import generate_scientific_name, generate_prompt, generate_description
from imageGenerateUtils import get_image, add_caption

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken?
A possible explanation is you have a new CUDA version which isn't
yet compatible with FA2? Please file a ticket to Unsloth or FA2.
We shall now use Xformers instead, which does not have any performance hits!
We found this negligible impact by benchmarking on 1x A100.
🦥 Unsloth Zoo will now patch everything to make training faster!


RuntimeError: Failed to import diffusers.pipelines.stable_diffusion_xl.pipeline_stable_diffusion_xl because of the following error (look up to see its traceback):
Failed to import diffusers.loaders.single_file because of the following error (look up to see its traceback):
name 'logger' is not defined

In [ ]:
# model setup

# download the stable diffusion model to your local, then specify the name here
stable_pretrained_model_link_or_path = "plantMilkModelSuite_flax.safetensors"

# specify text gen model name
text_model_name = "unsloth/Qwen3.5-35B-A3B-Instruct"
chat_template = "qwen-2.5"

monster_name_filepath = 'monsterNames.txt'

title_font = ImageFont.truetype("ipagp.ttf", 27)
paragraph_font = ImageFont.truetype("ipagp.ttf", 15)
caption_font = ImageFont.truetype("ipagp.ttf", 12)

In [ ]:
pipe = StableDiffusionXLPipeline.from_single_file(
    pretrained_model_link_or_path=stable_pretrained_model_link_or_path,
    torch_dtype=torch.float16
).to(device="cuda")

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name = text_model_name,
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    device_map = "auto"
)

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none"
)

In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = chat_template,
)

In [ ]:
name_generator = MarkovMonsterNameGenerator(n=2)
name_generator.train_from_file(monster_name_filepath)

In [ ]:
fields = [
    "杉林", "古代林", "畑", "草むら", "花畑", "密林", "水没林","ジャングル","峠","山の麓","樹海","竹林","森","霧の森","熱帯雨林","サバンナ","桜並木","果樹園",
    "洞窟", "鍾乳洞","谷底","岩石地帯","鉱山","荒野","岩の中",
    "雪原","凍土","氷河",
    "旧市街地", "化学工場跡地","都市の下水道", "古城","都市部","廃工場","地下鉄廃線","空中都市",
    "大砂漠", "オアシス",
    "海", "深海", "浅瀬", "砂浜", "汽水域", "川底","孤島","海底遺跡","湖","潮溜まり","地下水路","滝","沈没船","サンゴ礁",
    "成層圏","惑星中心部","溶岩地帯",
    "モンスターの体内"
]
spicies = [
    "生物",
    "鳥",
    "虫",
    "植物",
    "花",
    "草",
    "木",
    "キノコ",
    "魚",
    "爬虫類",
    "哺乳類",
    "両生類",
    "巨大生物",
    "小型生物",
    "草食動物",
    "肉食動物",
    "寄生生物",
    "絶滅危惧種",
    "甲殻類",
    "貝",
    "群生生物",
    "原始生物",
    "人工生命",
    "分類不明の生物"
]

In [ ]:
finalImages = []
for j in tqdm(range(25)):
    name = name_generator.generate()
    field = random.choice(fields)
    if field == "モンスターの体内":
        field = name_generator.generate()+"の体内"
    spicy = random.choice(spicies)
    if spicy in ("貝","草","鳥","魚") and random.randint(0,1)==1:
        name = name + spicy
    target = "{0}にて観測される架空の{1}「{2}」".format(field, spicy, name)
    print(target)
    description = generate_description(target, model, tokenizer)
    prompt = generate_prompt(target, description, model, tokenizer, pipe=pipe)
    scientific_name = generate_scientific_name(target, description, model, tokenizer)
    background = get_image(prompt.strip(), pipe)

    finalImage = add_caption(name, description, scientific_name, background, title_font, paragraph_font, caption_font)

    finalImage.save("endemic/{0}-{1}.png".format(j,name))
    finalImages.append(finalImage)

In [ ]:
fig, axes = plt.subplots(5, 5, figsize=(40,24))
plt.subplots_adjust(wspace=0.1, hspace=0.1)
for ax, img in tqdm(zip(axes.flatten(), finalImages)):
    ax.imshow(img)
    ax.axis('off')